# Flujo de Ejecucion - Diagramas de Control de Flujo

Este notebook explora la generacion de diagramas de flujo de ejecucion para algoritmos,
mostrando el camino paso a paso que sigue un programa durante su ejecucion.

## Configuracion Inicial

In [ ]:
import sys
sys.path.insert(0, '../..')

from pathlib import Path

from app.core.parser.pseudocode_parser import PseudocodeParser
from app.core.visualization.execution_flow_generator import (
    ExecutionFlowGenerator,
    FlowNode,
    FlowEdge,
    FlowNodeType,
    ExecutionFlowResult
)
from app.core.visualization.diagram_renderer import DiagramRenderer, RenderFormat
from app.core.visualization.graph_generator import GraphGenerator

# Directorio de salida
output_dir = Path("../../data/exports/notebooks/execution_flow")
output_dir.mkdir(parents=True, exist_ok=True)

print("Modulos importados correctamente")

---

## 1. Fundamentos del Flujo de Ejecucion

Los diagramas de flujo de ejecucion representan visualmente como se ejecuta un algoritmo,
mostrando las decisiones, bucles y operaciones en orden secuencial.

### 1.1 Tipos de Nodos

El generador de flujo soporta los siguientes tipos de nodos:

| Tipo | Descripcion | Forma |
|------|-------------|-------|
| START | Inicio del algoritmo | Ovalo |
| END | Fin del algoritmo | Ovalo |
| PROCESS | Operacion o asignacion | Rectangulo |
| DECISION | Condicion (if/while) | Diamante |
| LOOP_START | Inicio de bucle | Hexagono |
| LOOP_END | Fin de bucle | Hexagono |
| CALL | Llamada a funcion | Rectangulo con lineas |
| RETURN | Retorno de valor | Rectangulo redondeado |

In [ ]:
# Ver todos los tipos de nodos disponibles
print("Tipos de nodos disponibles:")
for node_type in FlowNodeType:
    print(f"  - {node_type.value}")

---

## 2. Ejemplo Basico: Algoritmo Secuencial

Comenzamos con un algoritmo simple sin estructuras de control.

In [ ]:
# Algoritmo secuencial simple
sequential_code = """
ALGORITHM CalcularArea(base, altura)
    area := base * altura
    area := area / 2
    RETURN area
END
"""

# Parsear
parser = PseudocodeParser()
ast = parser.parse(sequential_code)

print("AST generado correctamente")
print(f"Algoritmo: {ast.algorithm.name}")

### 2.1 Generar Flujo de Ejecucion

In [ ]:
# Crear generador
flow_generator = ExecutionFlowGenerator()

# Generar flujo
flow_result = flow_generator.generate(ast)

print(f"Flujo generado:")
print(f"  - Nodos: {len(flow_result.nodes)}")
print(f"  - Aristas: {len(flow_result.edges)}")
print(f"  - Nodo inicial: {flow_result.start_node}")
print(f"  - Nodos finales: {flow_result.end_nodes}")

### 2.2 Inspeccionar Nodos

In [ ]:
print("\nNodos del flujo:")
print("-" * 50)

for node in flow_result.nodes:
    print(f"[{node.id}] {node.node_type.value}: {node.label}")

### 2.3 Inspeccionar Aristas

In [ ]:
print("\nAristas del flujo:")
print("-" * 50)

for edge in flow_result.edges:
    label = f" ({edge.label})" if edge.label else ""
    print(f"  {edge.source} --> {edge.target}{label}")

---

## 3. Estructuras Condicionales

### 3.1 Estructura IF Simple

In [ ]:
if_code = """
ALGORITHM Maximo(a, b)
    IF a > b THEN
        max := a
    ELSE
        max := b
    ENDIF
    RETURN max
END
"""

ast_if = parser.parse(if_code)
flow_if = flow_generator.generate(ast_if)

print(f"Flujo con IF:")
print(f"  - Nodos: {len(flow_if.nodes)}")
print(f"  - Aristas: {len(flow_if.edges)}")

# Mostrar nodos de decision
decision_nodes = [n for n in flow_if.nodes if n.node_type == FlowNodeType.DECISION]
print(f"\nNodos de decision: {len(decision_nodes)}")
for node in decision_nodes:
    print(f"  - {node.label}")

### 3.2 Estructura IF Anidada

In [ ]:
nested_if_code = """
ALGORITHM ClasificarNumero(n)
    IF n > 0 THEN
        IF n > 100 THEN
            categoria := "grande"
        ELSE
            categoria := "pequeno"
        ENDIF
    ELSE
        IF n < 0 THEN
            categoria := "negativo"
        ELSE
            categoria := "cero"
        ENDIF
    ENDIF
    RETURN categoria
END
"""

ast_nested = parser.parse(nested_if_code)
flow_nested = flow_generator.generate(ast_nested)

print(f"Flujo con IF anidado:")
print(f"  - Nodos: {len(flow_nested.nodes)}")
print(f"  - Aristas: {len(flow_nested.edges)}")

# Contar nodos por tipo
from collections import Counter
type_counts = Counter(n.node_type.value for n in flow_nested.nodes)

print("\nDistribucion de nodos:")
for node_type, count in sorted(type_counts.items()):
    print(f"  - {node_type}: {count}")

---

## 4. Estructuras de Bucle

### 4.1 Bucle FOR

In [ ]:
for_code = """
ALGORITHM Sumatoria(n)
    suma := 0
    FOR i := 1 TO n DO
        suma := suma + i
    ENDFOR
    RETURN suma
END
"""

ast_for = parser.parse(for_code)
flow_for = flow_generator.generate(ast_for)

print(f"Flujo con FOR:")
print(f"  - Nodos: {len(flow_for.nodes)}")
print(f"  - Aristas: {len(flow_for.edges)}")

# Mostrar nodos de bucle
loop_nodes = [n for n in flow_for.nodes if 'loop' in n.node_type.value.lower()]
print(f"\nNodos de bucle: {len(loop_nodes)}")
for node in loop_nodes:
    print(f"  - [{node.node_type.value}] {node.label}")

### 4.2 Bucle WHILE

In [ ]:
while_code = """
ALGORITHM BusquedaBinaria(A, x, inicio, fin)
    WHILE inicio <= fin DO
        medio := (inicio + fin) / 2
        IF A[medio] = x THEN
            RETURN medio
        ENDIF
        IF A[medio] < x THEN
            inicio := medio + 1
        ELSE
            fin := medio - 1
        ENDIF
    ENDWHILE
    RETURN -1
END
"""

ast_while = parser.parse(while_code)
flow_while = flow_generator.generate(ast_while)

print(f"Flujo con WHILE:")
print(f"  - Nodos: {len(flow_while.nodes)}")
print(f"  - Aristas: {len(flow_while.edges)}")

### 4.3 Bucles Anidados

In [ ]:
nested_loop_code = """
ALGORITHM BubbleSort(A, n)
    FOR i := 1 TO n - 1 DO
        FOR j := 1 TO n - i DO
            IF A[j] > A[j + 1] THEN
                temp := A[j]
                A[j] := A[j + 1]
                A[j + 1] := temp
            ENDIF
        ENDFOR
    ENDFOR
END
"""

ast_bubble = parser.parse(nested_loop_code)
flow_bubble = flow_generator.generate(ast_bubble)

print(f"Flujo de BubbleSort:")
print(f"  - Nodos: {len(flow_bubble.nodes)}")
print(f"  - Aristas: {len(flow_bubble.edges)}")

# Estadisticas
stats = flow_bubble.statistics
print(f"\nEstadisticas:")
for key, value in stats.items():
    print(f"  - {key}: {value}")

---

## 5. Renderizado de Diagramas

### 5.1 Formato Mermaid

Mermaid es ideal para documentacion y visualizacion en navegadores.

In [ ]:
renderer = DiagramRenderer()

# Renderizar flujo simple en Mermaid
mermaid_output = renderer.render_flow(
    flow_result,
    format=RenderFormat.MERMAID
)

print("Diagrama Mermaid generado:")
print("-" * 50)
print(mermaid_output.content[:500] if len(mermaid_output.content) > 500 else mermaid_output.content)

### 5.2 Formato DOT (Graphviz)

DOT permite renderizado de alta calidad con Graphviz.

In [ ]:
dot_output = renderer.render_flow(
    flow_if,
    format=RenderFormat.DOT,
    options={
        'rankdir': 'TB',
        'node_shape': 'box',
        'edge_style': 'solid'
    }
)

print("Diagrama DOT generado:")
print("-" * 50)
print(dot_output.content[:500] if len(dot_output.content) > 500 else dot_output.content)

### 5.3 Formato JSON

El formato JSON es util para integracion con otras herramientas.

In [ ]:
json_output = renderer.render_flow(
    flow_for,
    format=RenderFormat.JSON
)

# Parsear y mostrar estructura
import json
flow_data = json.loads(json_output.content)

print("Estructura JSON:")
print(f"  - nodes: {len(flow_data.get('nodes', []))} elementos")
print(f"  - edges: {len(flow_data.get('edges', []))} elementos")
print(f"  - start_node: {flow_data.get('start_node')}")

---

## 6. Exportacion de Diagramas

### 6.1 Guardar en Archivos

In [ ]:
# Guardar diagrama en diferentes formatos
algorithms = [
    ("secuencial", flow_result),
    ("condicional", flow_if),
    ("bucle_for", flow_for),
    ("bubble_sort", flow_bubble)
]

for name, flow in algorithms:
    # Mermaid
    mermaid = renderer.render_flow(flow, format=RenderFormat.MERMAID)
    mermaid_path = output_dir / f"{name}_flow.mmd"
    mermaid_path.write_text(mermaid.content, encoding='utf-8')
    
    # DOT
    dot = renderer.render_flow(flow, format=RenderFormat.DOT)
    dot_path = output_dir / f"{name}_flow.dot"
    dot_path.write_text(dot.content, encoding='utf-8')
    
    # JSON
    json_out = renderer.render_flow(flow, format=RenderFormat.JSON)
    json_path = output_dir / f"{name}_flow.json"
    json_path.write_text(json_out.content, encoding='utf-8')
    
    print(f"Exportado: {name}")

print(f"\nArchivos guardados en: {output_dir}")

---

## 7. Analisis del Flujo

### 7.1 Complejidad Ciclomatica

La complejidad ciclomatica mide el numero de caminos linealmente independientes.

In [ ]:
def calcular_complejidad_ciclomatica(flow: ExecutionFlowResult) -> int:
    """
    Calcula la complejidad ciclomatica: M = E - N + 2P
    donde E = aristas, N = nodos, P = componentes conectados (1 para grafos conectados)
    """
    E = len(flow.edges)
    N = len(flow.nodes)
    P = 1  # Asumimos grafo conectado
    
    return E - N + 2 * P

# Calcular para cada algoritmo
print("Complejidad Ciclomatica:")
print("-" * 40)

for name, flow in algorithms:
    cc = calcular_complejidad_ciclomatica(flow)
    print(f"  {name}: {cc}")

### 7.2 Analisis de Caminos

In [ ]:
def contar_nodos_decision(flow: ExecutionFlowResult) -> int:
    """Cuenta nodos de decision en el flujo"""
    return sum(1 for n in flow.nodes if n.node_type == FlowNodeType.DECISION)

def contar_bucles(flow: ExecutionFlowResult) -> int:
    """Cuenta estructuras de bucle"""
    return sum(1 for n in flow.nodes if n.node_type == FlowNodeType.LOOP_START)

def profundidad_anidamiento(flow: ExecutionFlowResult) -> int:
    """Estima la profundidad maxima de anidamiento"""
    # Simplificacion: contar nodos de inicio de bucle/decision consecutivos
    max_depth = 0
    current_depth = 0
    
    for node in flow.nodes:
        if node.node_type in [FlowNodeType.DECISION, FlowNodeType.LOOP_START]:
            current_depth += 1
            max_depth = max(max_depth, current_depth)
        elif node.node_type in [FlowNodeType.LOOP_END]:
            current_depth = max(0, current_depth - 1)
    
    return max_depth

print("\nAnalisis de Flujo:")
print("-" * 50)

for name, flow in algorithms:
    print(f"\n{name}:")
    print(f"  - Nodos de decision: {contar_nodos_decision(flow)}")
    print(f"  - Bucles: {contar_bucles(flow)}")
    print(f"  - Profundidad estimada: {profundidad_anidamiento(flow)}")

---

## 8. Comparacion con Arboles de Recursion

Mientras los arboles de recursion muestran las llamadas recursivas,
los diagramas de flujo muestran la ejecucion secuencial.

In [ ]:
# Algoritmo iterativo vs recursivo

# Version iterativa
factorial_iter = """
ALGORITHM FactorialIterativo(n)
    resultado := 1
    FOR i := 1 TO n DO
        resultado := resultado * i
    ENDFOR
    RETURN resultado
END
"""

# Version recursiva (para comparar)
factorial_rec = """
ALGORITHM FactorialRecursivo(n)
    IF n <= 1 THEN
        RETURN 1
    ENDIF
    RETURN n * FactorialRecursivo(n - 1)
END
"""

# Generar flujos
ast_iter = parser.parse(factorial_iter)
ast_rec = parser.parse(factorial_rec)

flow_iter = flow_generator.generate(ast_iter)
flow_rec = flow_generator.generate(ast_rec)

print("Comparacion Factorial:")
print("-" * 40)
print(f"{'Metrica':<25} {'Iterativo':<12} {'Recursivo':<12}")
print("-" * 40)
print(f"{'Nodos':<25} {len(flow_iter.nodes):<12} {len(flow_rec.nodes):<12}")
print(f"{'Aristas':<25} {len(flow_iter.edges):<12} {len(flow_rec.edges):<12}")
print(f"{'Decisiones':<25} {contar_nodos_decision(flow_iter):<12} {contar_nodos_decision(flow_rec):<12}")
print(f"{'Bucles':<25} {contar_bucles(flow_iter):<12} {contar_bucles(flow_rec):<12}")

---

## 9. Integracion con Otros Modulos

### 9.1 Generar Grafo desde Flujo

In [ ]:
graph_generator = GraphGenerator()

# Convertir flujo a grafo para visualizacion
nodes_data = [
    {
        'id': node.id,
        'label': node.label,
        'type': node.node_type.value,
        'shape': _get_shape_for_type(node.node_type)
    }
    for node in flow_if.nodes
]

edges_data = [
    (edge.source, edge.target, {'label': edge.label or ''})
    for edge in flow_if.edges
]

def _get_shape_for_type(node_type: FlowNodeType) -> str:
    """Mapea tipo de nodo a forma visual"""
    shapes = {
        FlowNodeType.START: 'ellipse',
        FlowNodeType.END: 'ellipse',
        FlowNodeType.PROCESS: 'box',
        FlowNodeType.DECISION: 'diamond',
        FlowNodeType.LOOP_START: 'hexagon',
        FlowNodeType.LOOP_END: 'hexagon',
        FlowNodeType.CALL: 'box',
        FlowNodeType.RETURN: 'box'
    }
    return shapes.get(node_type, 'box')

print(f"Grafo generado con {len(nodes_data)} nodos y {len(edges_data)} aristas")

### 9.2 Conversion a Diccionario

In [ ]:
# Convertir resultado a diccionario para serializacion
flow_dict = flow_bubble.to_dict()

print("Estructura del resultado:")
print(f"  - nodes: {len(flow_dict['nodes'])} elementos")
print(f"  - edges: {len(flow_dict['edges'])} elementos")
print(f"  - start_node: {flow_dict['start_node']}")
print(f"  - statistics: {list(flow_dict['statistics'].keys())}")

---

## 10. Casos de Uso Avanzados

### 10.1 Algoritmos de Ordenamiento

In [ ]:
quicksort_code = """
ALGORITHM QuickSort(A, low, high)
    IF low < high THEN
        pivot := Partition(A, low, high)
        CALL QuickSort(A, low, pivot - 1)
        CALL QuickSort(A, pivot + 1, high)
    ENDIF
END
"""

ast_qs = parser.parse(quicksort_code)
flow_qs = flow_generator.generate(ast_qs)

print("Flujo de QuickSort:")
print(f"  - Nodos: {len(flow_qs.nodes)}")
print(f"  - Aristas: {len(flow_qs.edges)}")

# Identificar llamadas recursivas
call_nodes = [n for n in flow_qs.nodes if n.node_type == FlowNodeType.CALL]
print(f"\nLlamadas a funciones: {len(call_nodes)}")
for node in call_nodes:
    print(f"  - {node.label}")

### 10.2 Algoritmos de Busqueda

In [ ]:
linear_search_code = """
ALGORITHM BusquedaLineal(A, n, x)
    FOR i := 1 TO n DO
        IF A[i] = x THEN
            RETURN i
        ENDIF
    ENDFOR
    RETURN -1
END
"""

ast_search = parser.parse(linear_search_code)
flow_search = flow_generator.generate(ast_search)

print("Flujo de Busqueda Lineal:")
print(f"  - Nodos: {len(flow_search.nodes)}")
print(f"  - Tiene retorno temprano: {any(n.node_type == FlowNodeType.RETURN for n in flow_search.nodes)}")

---

## Resumen

En este notebook cubrimos:

1. **Fundamentos**: Tipos de nodos y estructura del flujo
2. **Algoritmos secuenciales**: Flujo lineal simple
3. **Condicionales**: IF, IF-ELSE, IF anidados
4. **Bucles**: FOR, WHILE, bucles anidados
5. **Renderizado**: Mermaid, DOT, JSON
6. **Exportacion**: Guardar en multiples formatos
7. **Analisis**: Complejidad ciclomatica, caminos
8. **Comparacion**: Iterativo vs recursivo
9. **Integracion**: Con otros modulos de visualizacion
10. **Casos avanzados**: Ordenamiento, busqueda

El modulo `ExecutionFlowGenerator` es util para:
- Documentacion de algoritmos
- Analisis de complejidad estructural
- Generacion de diagramas de flujo estandar
- Integracion con herramientas de visualizacion

In [ ]:
print("Notebook completado")
print(f"Archivos generados en: {output_dir}")